# 코스피 방향 예측 — 로지스틱 회귀 (해답 보강판)

이번 달 거시·금융 지표 29개로 **다음 달 코스피가 오를지(1) 내릴지(0)** 를 분류합니다. 원본 *코스피지수_로짓해답_wget.ipynb*를 정리하고 평가를 보강했습니다.

| 파트 | 내용 |
|---|---|
| **A** | 원본 실습: 목표변수 만들기 → L1 로지스틱 회귀 → 정확도 |
| **B** | 보강: 혼동행렬·정밀도·재현율·F1·ROC AUC, 기준선, 표준화, 무작위 분할 반복, 임계값 |
| **C** | 강의 예제 재현: 비만 8마리 ROC(AUC 0.906), 붓꽃 3품종 |

**원본 대비 달라진 점**
1. 데이터는 강의 1일차와 같은 `KOSPI_Index_KO.csv`(UTF-8, `날짜` 열 포함)를 씁니다.
2. 목표변수를 만드는 셀을 **여러 번 실행해도 결과가 같도록** 바꿨습니다. 원본은 `kospi = kospi[:-1]`을 다시 실행하면 행이 계속 줄어듭니다. 저장된 출력(157행, 정확도 0.525)은 첫 두 행이 빠진 상태의 결과이고, 처음부터 실행하면 159행, 정확도 0.55입니다.
3. 정확도 하나만 보지 않고 **기준선과 비교**하고, **무작위 분할 100번**으로 다시 확인합니다.

## 0. 준비

In [1]:
import os
FILE = 'KOSPI_Index_KO.csv'
if not os.path.exists(FILE):
    try:
        from google.colab import files
        print('업로드할 파일:', FILE)
        files.upload()
    except ImportError:
        raise FileNotFoundError(f'{os.getcwd()}에 {FILE}이 없습니다')

In [2]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings('ignore')
plt.rcParams['axes.unicode_minus'] = False
print('scikit-learn', sklearn.__version__)

def l1_logit(C):
    """L1 로지스틱 회귀. scikit-learn 1.8부터 penalty 인자가 l1_ratio로 바뀐 것을 반영"""
    major, minor = map(int, sklearn.__version__.split('.')[:2])
    if (major, minor) >= (1, 8):
        return LogisticRegression(solver='liblinear', l1_ratio=1, C=C, random_state=0)
    return LogisticRegression(solver='liblinear', penalty='l1', C=C, random_state=0)

scikit-learn 1.8.0


# 파트 A — 원본 실습

## 1. 데이터 불러오기

In [3]:
kospi = pd.read_csv(FILE, encoding='utf-8-sig')
print(kospi.shape)
kospi.head()

(160, 31)


,날짜,전경련BSI,주택매매가격,건설BSI(전망),"실업률(계절조정,%)",어음부도율,일별수출액YoY,내국인출국자수YoY,코스피 200 변동성지수,통안채 364일,...,재화수출,내수,GDP디플레이터,총저축률,총투자율,Reuter CRB(EW)상품선물지수,Reuter CRB(EW)에너지지수,Reuter CRB(EW)산업지수,Reuter CRB(EW)귀금속지수,코스피지수
0,2003-05,96.4,70.13,83.3,3.7,0.09,14.50,-37.03,25.50,4.06,...,1.6,-0.4,3.1,32.8,32.3,243.7,247.7,217.5,322.4,633.4
1,2003-06,90.3,70.83,86.7,3.7,0.08,10.56,-9.67,20.83,4.24,...,7.8,-0.4,4.0,33.3,32.3,247.6,255.7,247.3,326.4,669.9
2,2003-07,91.4,70.57,86.7,3.8,0.06,14.83,0.93,26.98,4.60,...,7.8,0.3,4.0,33.3,31.8,249.9,268.7,232.3,340.1,713.5
3,2003-08,109.6,69.99,92.4,3.9,0.06,24.13,2.85,28.33,4.66,...,7.8,0.3,4.0,33.3,31.8,255.3,283.3,256.6,364.1,759.5
4,2003-09,110.3,69.70,64.1,3.8,0.08,25.49,14.34,25.35,4.51,...,10.9,0.3,3.3,34.7,31.8,262.6,292.7,262.0,368.3,697.5


## 2. 목표변수: 다음 달에 오르면 1

한 행은 t월 지표와 t월에서 t+1월로 가는 방향을 짝지은 것입니다. 특성에 다른 달의 값을 쓰지 않으므로 각 행은 독립 관측치이고, 무작위로 섞어 나누는 것이 맞습니다. 원본 셀을 한 번에 새 데이터프레임으로 만들도록 바꿔, 여러 번 실행해도 같은 결과가 나옵니다.

In [4]:
def classify(current, future):
    if future > current:
        return 1
    else:
        return 0

df = kospi.copy()                                   # 원본은 그대로 두고 복사본에서 작업
df['다음달'] = df['코스피지수'].shift(-1)
df = df.iloc[:-1].copy()                            # 다음 달 값이 없는 마지막 달 제외
df['target'] = list(map(classify, df['코스피지수'], df['다음달']))
df[['날짜', '코스피지수', '다음달', 'target']].head()

,날짜,코스피지수,다음달,target
0,2003-05,633.4,669.9,1
1,2003-06,669.9,713.5,1
2,2003-07,713.5,759.5,1
3,2003-08,759.5,697.5,0
4,2003-09,697.5,782.4,1


In [5]:
X = df.drop(columns=['날짜', '코스피지수', '다음달', 'target'])
y = df['target']
print(X.shape)
print(y.value_counts().rename({1: '상승', 0: '하락·보합'}))
print('상승 비율: %.3f' % y.mean())

(159, 29)
target
상승       91
하락·보합    68
Name: count, dtype: int64
상승 비율: 0.572


## 3. 분할과 적합 (원본 설정)

In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)   # 기본 75/25, 무작위
lr = l1_logit(C=10)
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

In [7]:
from sklearn.metrics import accuracy_score
print('정확도:', accuracy_score(y_test, y_pred))

정확도: 0.55


# 파트 B — 평가 보강

## 4. 혼동행렬과 성과지표

In [8]:
from sklearn.metrics import (confusion_matrix, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report)
print('혼동행렬 (행 = 실제 [하락, 상승], 열 = 예측 [하락, 상승])')
print(confusion_matrix(y_test, y_pred))
print('정밀도: %.3f' % precision_score(y_test, y_pred))
print('재현율: %.3f' % recall_score(y_test, y_pred))
print('F1    : %.3f' % f1_score(y_test, y_pred))
proba = lr.predict_proba(X_test)[:, 1]            # AUC는 확률로 계산한다
print('ROC AUC: %.3f' % roc_auc_score(y_test, proba))

혼동행렬 (행 = 실제 [하락, 상승], 열 = 예측 [하락, 상승])
[[ 9 11]
 [ 7 13]]
정밀도: 0.542
재현율: 0.650
F1    : 0.591
ROC AUC: 0.555


In [9]:
print(classification_report(y_test, y_pred, target_names=['하락', '상승']))

              precision    recall  f1-score   support

          하락       0.56      0.45      0.50        20
          상승       0.54      0.65      0.59        20

    accuracy                           0.55        40
   macro avg       0.55      0.55      0.55        40
weighted avg       0.55      0.55      0.55        40



## 5. 함정 1: 기준선과 비교하기

'항상 상승'이라고만 해도 테스트 세트의 상승 비율만큼은 맞습니다.

In [10]:
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
print('항상 다수 클래스(상승) 정확도:', accuracy_score(y_test, dummy.predict(X_test)))
print('테스트 세트 상승 비율       :', y_test.mean())

항상 다수 클래스(상승) 정확도: 0.5
테스트 세트 상승 비율       : 0.5


## 6. 함정 2: 표준화 없이 L1 규제

벌점은 계수 크기에 걸리므로, 단위가 큰 특성(출국자 수 등)은 계수가 작아져 벌점을 덜 받습니다. 파이프라인으로 표준화를 묶습니다.

In [11]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
pipe = make_pipeline(StandardScaler(), l1_logit(C=10)).fit(X_train, y_train)
print('표준화 후 정확도: %.3f' % accuracy_score(y_test, pipe.predict(X_test)))
print('표준화 후 AUC   : %.3f' % roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1]))

표준화 후 정확도: 0.550
표준화 후 AUC   : 0.565


## 7. 함정 3: 무작위 분할 한 번만 믿기

테스트 세트가 40개월뿐이라 분할 한 번은 표본 하나에 불과합니다. 무작위 75/25 분할을 100번(seed 0~99) 반복하고, 같은 분할에서 '항상 상승'과 비교합니다.

In [12]:
rows = []
for seed in range(100):
    Xa, Xb, ya, yb = train_test_split(X, y, random_state=seed)
    m = make_pipeline(StandardScaler(), l1_logit(C=10)).fit(Xa, ya)
    rows.append({'정확도': accuracy_score(yb, m.predict(Xb)),
                 'AUC': roc_auc_score(yb, m.predict_proba(Xb)[:, 1]),
                 '항상 상승': yb.mean()})
rep = pd.DataFrame(rows)
print(rep.describe().loc[['mean', 'std', 'min', 'max']].round(3))
print('항상 상승을 이긴 분할:', (rep['정확도'] > rep['항상 상승']).sum(), '/ 100')
print('AUC가 0.5를 넘은 분할:', (rep['AUC'] > 0.5).sum(), '/ 100')

        정확도    AUC  항상 상승
mean  0.545  0.555  0.570
std   0.076  0.080  0.066
min   0.375  0.387  0.450
max   0.725  0.752  0.725
항상 상승을 이긴 분할: 40 / 100
AUC가 0.5를 넘은 분할: 73 / 100


### C를 반복 층화 교차검증으로 고르기

In [13]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=20, random_state=0)
for C in [0.01, 0.1, 1, 10]:
    m = make_pipeline(StandardScaler(), l1_logit(C))
    auc = cross_val_score(m, X, y, cv=cv, scoring='roc_auc')
    print(f'C={C:<5} AUC 평균 {auc.mean():.3f}  범위 {auc.min():.2f}~{auc.max():.2f}')

C=0.01  AUC 평균 0.500  범위 0.50~0.50


C=0.1   AUC 평균 0.464  범위 0.30~0.71


C=1     AUC 평균 0.580  범위 0.34~0.77


C=10    AUC 평균 0.562  범위 0.31~0.75


3절의 0.55는 표본 하나입니다. 무작위 분할 100번에서 정확도 평균은 0.545로 항상 상승(0.570)보다 낮고, 모델이 항상 상승을 이긴 분할은 40번뿐입니다. 가장 나은 C를 골라도 AUC 평균은 0.58 안팎입니다. 거시 지표 29개로도 다음 달 코스피 방향을 안정적으로 맞히지 못한다는 뜻입니다. 정확도 55%가 의미를 가지려면 (1) 기준선을 넘고, (2) 다른 무작위 분할에서도 반복되고, (3) 거래비용 뒤에도 남아야 합니다.

## 8. 임계값 바꿔 보기

`predict`는 임계값 0.5를 씁니다. 확률을 직접 받아 임계값을 바꾸면 혼동행렬이 달라집니다. 6절에서 표준화한 모델을 같은 테스트 세트에 그대로 씁니다.

In [14]:
p = pipe.predict_proba(X_test)[:, 1]
for t in [0.3, 0.5, 0.7]:
    pred = (p >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    print(f'임계값 {t}: TP={tp:2d} FP={fp:2d} FN={fn:2d} TN={tn:2d}  재현율={tp/(tp+fn):.2f}  정밀도={tp/max(tp+fp,1):.2f}')

임계값 0.3: TP=18 FP=13 FN= 2 TN= 7  재현율=0.90  정밀도=0.58
임계값 0.5: TP=13 FP=11 FN= 7 TN= 9  재현율=0.65  정밀도=0.54
임계값 0.7: TP=10 FP=10 FN=10 TN=10  재현율=0.50  정밀도=0.50


# 파트 C — 강의 예제 재현

## 9. 비만 8마리 ROC (강의의 AUC 0.9)

In [15]:
from sklearn.metrics import roc_curve
y_mice = np.array([0, 0, 0, 0, 1, 1, 1, 1])                       # 0 = 정상, 1 = 비만
score  = np.array([0.03, 0.06, 0.30, 0.55, 0.30, 0.85, 0.93, 0.97])   # 모델이 준 비만 확률
fpr, tpr, thr = roc_curve(y_mice, score)
print(pd.DataFrame({'임계값': thr, 'FPR': fpr, 'TPR': tpr}).round(3))
print('AUC =', roc_auc_score(y_mice, score))

    임계값   FPR   TPR
0   inf  0.00  0.00
1  0.97  0.00  0.25
2  0.85  0.00  0.75
3  0.55  0.25  0.75
4  0.30  0.50  1.00
5  0.03  1.00  1.00
AUC = 0.90625


## 10. 붓꽃 3품종 분류 (강의 실습 1)

In [16]:
from sklearn.datasets import load_iris
iris = load_iris(as_frame=True)
Xi_train, Xi_test, yi_train, yi_test = train_test_split(iris.data, iris.target, test_size=0.2, random_state=1)
clf = LogisticRegression(max_iter=1000).fit(Xi_train, yi_train)     # 다중 클래스: softmax(다항)
yi_pred = clf.predict(Xi_test)
print('정확도: {:.2f}%'.format(accuracy_score(yi_test, yi_pred) * 100))
print('계수 모양:', clf.coef_.shape)
print(confusion_matrix(yi_test, yi_pred))
print('첫 샘플의 품종별 확률:', clf.predict_proba(Xi_test.iloc[:1]).round(3), '(합 = 1)')

정확도: 96.67%
계수 모양: (3, 4)
[[11  0  0]
 [ 0 12  1]
 [ 0  0  6]]
첫 샘플의 품종별 확률: [[0.985 0.015 0.   ]] (합 = 1)


## 11. 연습문제
1. 목표변수를 '다음 달 수익률이 −5% 이하'(급락)로 바꾸면 양성 비율은 얼마인가? 정확도 대신 어떤 지표를 봐야 하나?
2. `LogisticRegressionCV(cv=StratifiedKFold(5), Cs=10)`로 C를 고르고, 결과를 기준선과 비교하라.
3. 비만 8마리 예제에서 Youden 지수 J = TPR − FPR이 가장 큰 임계값은 무엇인가?